In [ ]:
import os
import cv2
import numpy as np
import nibabel as nib
from pathlib import Path
from scipy.io import loadmat

def sharp_preds2nii():
    # 1. Setup paths
    # Current directory is /util
    util_dir = Path.cwd()
    stats_file = util_dir / 'test_7T.mat'
    
    if not stats_file.exists():
        raise FileNotFoundError(f"Metadata file {stats_file} not found in util folder.")
    
    # Load Stats (max_val, min_val are usually stored as arrays in the .mat)
    mat_data = loadmat(str(stats_file))
    max_vals = mat_data['max_val'].flatten()
    min_vals = mat_data['min_val'].flatten()

    rep_suffix = 'L1' # needs to be adapted for other models (e.g. ablation/isolation study)
    
    # Path Logic: Replace 'util' with 'data' in the path
    data_root = Path(str(util_dir).replace('util', 'data'))
    base_folder = data_root / 'x2' / 'preds'
    out_folder = data_root / 'preds_nii'
    
    # Create output folder
    out_folder.mkdir(parents=True, exist_ok=True)

    # Find directories ending in L1
    folders_to_do = sorted([d for d in base_folder.iterdir() if d.is_dir() and d.name.endswith(rep_suffix)])

    for idx, folder in enumerate(folders_to_do):
        subject_name = folder.name[:6]  # Extract e.g. 100610
        print(f"Working on: {folder.name}")

        # 2. Retrieve original metadata (Header)
        original_nii_path = data_root / subject_name / 'T1w' / 'Diffusion_7T' / 'data.nii.gz'
        
        if original_nii_path.exists():
            orig_nifti = nib.load(str(original_nii_path))
            header = orig_nifti.header
            affine = orig_nifti.affine
        else:
            print(f"Warning: Original NIfTI not found for {subject_name}. Skipping...")
            continue

        # 3. Read PNGs into 4D volume
        # Get all pngs and sort them to match frame/slice order
        png_files = sorted(list(folder.glob('*.png')))
        
        if not png_files:
            print(f"Warning: No PNGs found in {folder}")
            continue

        # Preallocate (H, W, Slices, B-values) based on cropped size
        vol_data = np.zeros((206, 172, 173, 143), dtype=np.uint16)

        counter = 0
        for j in range(143):     # b-values
            for k in range(173): # slices
                if counter >= len(png_files):
                    break
                
                # cv2.IMREAD_UNCHANGED for 16-bit PNGs
                img = cv2.imread(str(png_files[counter]), cv2.IMREAD_UNCHANGED)
                vol_data[:, :, k, j] = img
                counter += 1

        # 4. Rescale to original intensities
        # Convert uint16 to float64 (0.0 to 1.0)
        rescaled = vol_data.astype(np.float64) / 65535.0
        
        # Apply scaling: (norm * (max - min)) + min
        m_max = max_vals[idx]
        m_min = min_vals[idx]
        rescaled = (rescaled * (m_max - m_min)) + m_min

        # 5. Zero Padding to 207x173 to match original dimensions
        # Create empty array of target size
        padded = np.zeros((207, 173, 173, 143), dtype=np.float32)
        padded[0:206, 0:172, :, :] = rescaled

        # 6. Format and Save    
        # to reverse the specific XY flip 
        final_data = np.transpose(padded, (1, 0, 2, 3))
        
        save_path = out_folder / f"{folder.name}.nii.gz"
        
        # Create NIfTI object
        new_nifti = nib.Nifti1Image(final_data, affine, header)
        nib.save(new_nifti, str(save_path))
        
        print(f"NIfTI restored with padding: {save_path.name} done!")

if __name__ == "__main__":
    sharp_preds2nii()